# Module 19 — Fine-tuning, LoRA, and Adapters on Mac

Companion notebook to [`docs/modules/19_finetuning_lora_and_adapters.md`](../docs/modules/19_finetuning_lora_and_adapters.md).

Real, deterministic Python for the decision framework, dataset creation/cleaning/splitting,
LoRA parameter math, overfitting detection, adapter metadata tracking, and a before/after
evaluation harness. Real LoRA training itself needs base model weights and Apple Silicon
compute this dev machine intentionally doesn't run a model on, so `mlx_lora.py`'s
training/merge calls are honest-skipped via dependency injection — everything else below runs
for real.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_19"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. The decision framework — prompting vs RAG vs fine-tuning

In [2]:
from local_ai_core.finetuning.decision_framework import recommend_approach

changing_knowledge = recommend_approach(needs_private_or_changing_knowledge=True)
ready_to_fine_tune = recommend_approach(
    output_style_is_repetitive=True,
    task_is_narrow_and_stable=True,
    has_enough_labeled_data=True,
    evaluation_proves_improvement=True,
)
simple_task = recommend_approach(task_is_simple=True)

for label, rec in [
    ("changing knowledge", changing_knowledge),
    ("fine-tuning ready", ready_to_fine_tune),
    ("simple task", simple_task),
]:
    print(f"{label}: {rec.approach.value} — {rec.reason}")

changing knowledge: rag — knowledge is private/changing or answers must cite sources - do not fine-tune to add facts that change; use RAG
fine-tuning ready: fine_tuning — output style is repetitive, the task is narrow and stable, enough labeled data exists, and evaluation proves improvement
simple task: prompting — task is simple and/or behavior can be fully specified in instructions


## 2. Lab 1: dataset creation, cleaning, splitting, leakage detection

In [3]:
from dataset_demo import run_lab as run_dataset_lab, result_to_markdown as dataset_to_markdown

dataset_result = run_dataset_lab()
print(dataset_to_markdown(dataset_result))

# Lab 1 - dataset creation, cleaning, splitting

- Raw examples (with one deliberate duplicate): 41
- Cleaned examples: 40 (dropped 1: ['duplicate instruction+input'])
- Split: train=32, validation=4, test=4
- Leakage check: 0 leaked (instruction, input) pairs



## 3. LoRA parameter math — a genuine, computable reduction

In [4]:
from local_ai_core.finetuning.lora_math import LayerShape, compare_full_finetune_and_lora

qwen_attention_layers = [
    LayerShape(name="q_proj", d_in=1536, d_out=1536),
    LayerShape(name="k_proj", d_in=1536, d_out=256),
    LayerShape(name="v_proj", d_in=1536, d_out=256),
    LayerShape(name="o_proj", d_in=1536, d_out=1536),
]

report = compare_full_finetune_and_lora(qwen_attention_layers, rank=8)
print(f"Full fine-tune trainable params: {report.full_finetune_params:,}")
print(f"LoRA (rank 8) trainable params: {report.lora_trainable_params:,}")
print(f"LoRA is {report.reduction_ratio:.2%} the size of full fine-tuning")

Full fine-tune trainable params: 5,505,024
LoRA (rank 8) trainable params: 77,824
LoRA is 1.41% the size of full fine-tuning


## 4. Overfitting detection over a real loss-curve algorithm

In [5]:
from local_ai_core.finetuning.overfitting import EpochLoss, detect_overfitting

# Synthetic loss curve (no real training runs on this machine) — illustrates
# the detector, not a real training result.
loss_curve = [
    EpochLoss(epoch=1, train_loss=1.20, validation_loss=1.25),
    EpochLoss(epoch=2, train_loss=0.85, validation_loss=0.95),
    EpochLoss(epoch=3, train_loss=0.60, validation_loss=0.90),
    EpochLoss(epoch=4, train_loss=0.40, validation_loss=1.00),
    EpochLoss(epoch=5, train_loss=0.25, validation_loss=1.15),
]

overfitting_report = detect_overfitting(loss_curve, patience=2)
print(f"Overfitting detected: {overfitting_report.is_overfitting}")
print(f"Would have stopped at epoch: {overfitting_report.stop_at_epoch}")
print(f"Best validation loss: {overfitting_report.best_validation_loss} (epoch {overfitting_report.best_validation_epoch})")

Overfitting detected: True
Would have stopped at epoch: 5
Best validation loss: 0.9 (epoch 3)


## 5. MLX fine-tuning path — honest-skip on this machine

In [6]:
from local_ai_core.finetuning.mlx_lora import MlxLoraTrainer, TrainingConfig, is_mlx_lm_available

print(f"mlx-lm importable on this machine: {is_mlx_lm_available()}")


def fake_train(config: TrainingConfig) -> int:
    print(f"(fake) training {config.base_model} on {config.dataset_dir} -> {config.adapter_output_dir}")
    return 0


trainer = MlxLoraTrainer(train_fn=fake_train)
training_result = trainer.train_lora_adapter(
    TrainingConfig(
        base_model="mlx-community/Qwen2.5-1.5B-Instruct-4bit",
        dataset_dir=str(REPO_ROOT / "datasets" / "finetuning"),
        adapter_output_dir="adapters/ticket-classifier-v1",
    )
)
print(f"Training succeeded: {training_result.succeeded}")

mlx-lm importable on this machine: False
(fake) training mlx-community/Qwen2.5-1.5B-Instruct-4bit on /Users/bhakti/workspace/local_ai_app/datasets/finetuning -> adapters/ticket-classifier-v1
Training succeeded: True


## 6. Labs 3-4: before/after evaluation against real golden cases

In [7]:
from evaluation_demo import run_lab as run_evaluation_lab, result_to_markdown as evaluation_to_markdown

evaluation_result = await run_evaluation_lab()
print(evaluation_to_markdown(evaluation_result))

# Labs 3-4 - before/after evaluation, comparison against baselines

## Lab 3: prompt-only baseline vs fine-tuned candidate
- baseline mean score: 0.50
- candidate mean score: 1.00
- delta: +0.50 (improved=True)

## Lab 4: unhelpful baseline vs fine-tuned candidate
- baseline mean score: 0.00
- candidate mean score: 1.00
- delta: +1.00 (improved=True)



## 7. Labs 5-6: package the adapter, document failure cases

In [8]:
from adapter_packaging_demo import run_lab as run_packaging_lab, result_to_markdown as packaging_to_markdown

packaging_result = run_packaging_lab()
print(packaging_to_markdown(packaging_result))

# Labs 5-6 - adapter packaging, failure case analysis

## Lab 5: package adapter
- Full fine-tune trainable params: 5,505,024
- LoRA (rank 8) trainable params: 77,824 (1.41% of full fine-tune)
- Dataset hash: ff75090691637244
- Registered adapter: ticket-classifier-v1 (created_at=2026-07-09 18:44:41)

## Lab 6: failure case analysis
- Overfitting detected: True
- Would have stopped at epoch: 5
- Best validation loss: 0.90 (epoch 3)
- Failure mode: validation loss climbs after epoch 2 while train loss keeps falling - the model is memorizing training tickets rather than learning the general category boundary. Real remediation: stop at the best-validation checkpoint, add more labeled examples per category, or lower the LoRA rank to reduce capacity to memorize.



## Summary

- Decision framework: a real, testable function replacing a bulleted list.
- Dataset: 40 real hand-labeled examples, cleaned and split with a real leakage check.
- LoRA math: a genuine parameter-count reduction for a realistic small-model shape.
- Overfitting: a real early-stopping signal over a (synthetic) loss curve.
- MLX path: honest-skipped on this machine, DI-testable, real subprocess wiring documented for
  a resourced Mac.
- Evaluation: a real before/after comparison harness reusing Module 13's `must_contain_score`.
- Packaging: a real SQLite adapter registry entry, with an honest failure-case writeup.